In [ ]:
# Example 1: Basic cached property usage
import propcache
import time
import random

class BasicExample:
    """Basic class demonstrating cached property functionality"""
    
    def __init__(self, name):
        self.name = name
        self._computation_count = 0
    
    @propcache.cached_property
    def expensive_property(self):
        """A property that simulates expensive computation"""
        self._computation_count += 1
        print(f"Computing expensive_property for {self.name} (call #{self._computation_count})")
        time.sleep(0.1)  # Simulate expensive operation
        return f"Computed result for {self.name}: {random.randint(1000, 9999)}"
    
    @propcache.cached_property
    def derived_value(self):
        """Property that depends on other cached properties"""
        base_value = len(self.name) * 100
        return base_value + hash(self.expensive_property) % 1000

print("Basic cached property demonstration")
print("=" * 38)

# Create instance and test caching behavior
obj1 = BasicExample("TestObject1")
print(f"Created object: {obj1.name}")

# First access - should compute
print(f"\nFirst access to expensive_property:")
result1 = obj1.expensive_property
print(f"Result: {result1}")

# Second access - should use cached value
print(f"\nSecond access to expensive_property:")
result2 = obj1.expensive_property
print(f"Result: {result2}")
print(f"Same result? {result1 == result2}")

# Access derived property
print(f"\nAccessing derived_value:")
derived = obj1.derived_value
print(f"Derived value: {derived}")

# Verify computation count: repeated access and the dependent property reuse the cached value
print(f"\nTotal computations: {obj1._computation_count}")
assert result1 == result2, (result1, result2)
assert obj1._computation_count == 1, obj1._computation_count
assert vars(obj1)["expensive_property"] == result1  # stored in the instance __dict__ under its own name

# Test with multiple instances
obj2 = BasicExample("AnotherObject")
print(f"\nCreated second object: {obj2.name}")
result3 = obj2.expensive_property
print(f"Second object result: {result3}")
print(f"Different from first? {result1 != result3}")
assert obj2._computation_count == 1 and obj1._computation_count == 1  # caching is per instance

print("✓ Basic cached property functionality verified!")

In [ ]:
# Example 2: Cache invalidation and property management
import propcache
import time
import threading

class ManagedCache:
    """Demonstrates cache invalidation and property management"""
    
    def __init__(self, base_value):
        self.base_value = base_value
        self._computation_calls = {}
    
    @propcache.cached_property
    def computed_hash(self):
        """Expensive hash computation based on base_value"""
        call_count = self._computation_calls.get('computed_hash', 0) + 1
        self._computation_calls['computed_hash'] = call_count
        print(f"Computing hash (call #{call_count})...")
        
        # Simulate expensive hash computation
        result = 0
        for char in str(self.base_value):
            result = (result * 31 + ord(char)) % (10**9 + 7)
            time.sleep(0.01)  # Simulate work
        return result
    
    @propcache.cached_property
    def formatted_result(self):
        """Property that depends on computed_hash"""
        call_count = self._computation_calls.get('formatted_result', 0) + 1
        self._computation_calls['formatted_result'] = call_count
        print(f"Computing formatted_result (call #{call_count})...")
        
        hash_val = self.computed_hash
        return f"Hash-{hash_val:08x}-{len(str(hash_val))}"
    
    def update_base_value(self, new_value):
        """Update base value and invalidate cached properties"""
        self.base_value = new_value
        
        # propcache.cached_property stores each value in the instance __dict__ under the
        # property's own name; removing that entry makes the next access recompute it.
        self.__dict__.pop('computed_hash', None)
        self.__dict__.pop('formatted_result', None)
        
        print(f"Base value updated to {new_value}, cache invalidated")
    
    def get_cache_info(self):
        """Names of the cached properties that currently hold a value"""
        return sorted(
            name for name in vars(self)
            if isinstance(type(self).__dict__.get(name), propcache.cached_property)
        )

print("Cache invalidation and property management")
print("=" * 42)

# Create managed cache instance
cache_obj = ManagedCache("initial_value_123")

# Test initial computation
print("Initial access to properties:")
hash_result1 = cache_obj.computed_hash
print(f"Computed hash: {hash_result1}")

formatted1 = cache_obj.formatted_result
print(f"Formatted result: {formatted1}")

# Test caching - should not recompute
print(f"\nSecond access (should use cache):")
hash_result2 = cache_obj.computed_hash
formatted2 = cache_obj.formatted_result
print(f"Hash cached? {hash_result1 == hash_result2}")
print(f"Formatted cached? {formatted1 == formatted2}")

# Check computation calls
print(f"Computation calls so far: {cache_obj._computation_calls}")
assert hash_result1 == hash_result2 and formatted1 == formatted2
assert cache_obj._computation_calls == {'computed_hash': 1, 'formatted_result': 1}, cache_obj._computation_calls
assert cache_obj.get_cache_info() == ['computed_hash', 'formatted_result'], cache_obj.get_cache_info()

# Test cache invalidation
print(f"\nUpdating base value and testing invalidation:")
cache_obj.update_base_value("new_value_456")
assert cache_obj.get_cache_info() == [], cache_obj.get_cache_info()

# Access after invalidation - should recompute
hash_result3 = cache_obj.computed_hash
formatted3 = cache_obj.formatted_result
print(f"New hash result: {hash_result3}")
print(f"New formatted result: {formatted3}")
print(f"Hash changed? {hash_result1 != hash_result3}")
print(f"Formatted changed? {formatted1 != formatted3}")

# Final computation count
print(f"Final computation calls: {cache_obj._computation_calls}")
assert cache_obj._computation_calls == {'computed_hash': 2, 'formatted_result': 2}, cache_obj._computation_calls
assert hash_result3 != hash_result1 and formatted3 != formatted1
assert cache_obj.get_cache_info() == ['computed_hash', 'formatted_result'], cache_obj.get_cache_info()

print("✓ Cache invalidation and property management verified!")

In [ ]:
# Example 3: Concurrent access from several threads
import propcache
import time
import threading
import random
from concurrent.futures import ThreadPoolExecutor

# propcache.cached_property takes no lock. Threads that read a cold property at the same
# time can each run the getter; the last value stored in the instance __dict__ wins, and
# from then on every read returns that stored value.
class SharedComputation:
    """Cached properties read by several threads at once"""

    def __init__(self, identifier):
        self.identifier = identifier
        self._log_lock = threading.Lock()
        self.computed_values = []

    @propcache.cached_property
    def expensive_computation(self):
        """Expensive computation with no locking of its own"""
        time.sleep(0.1)  # widen the window in which other threads also miss the cache
        result = random.randint(1, 10**9)
        with self._log_lock:
            self.computed_values.append(result)
        return result

    @propcache.cached_property
    def derived_property(self):
        """Property that depends on expensive_computation"""
        return self.expensive_computation * 2 + len(self.identifier)


class LockedComputation:
    """The same computation behind an explicit lock, so it runs exactly once"""

    def __init__(self):
        self._lock = threading.Lock()
        self.compute_count = 0

    @propcache.cached_property
    def expensive_computation(self):
        with self._lock:
            # Another thread may have finished the computation while this one waited.
            if "expensive_computation" in self.__dict__:
                return self.__dict__["expensive_computation"]
            self.compute_count += 1
            time.sleep(0.1)
            result = random.randint(1, 10**9)
            # Store it before releasing the lock, so the next thread finds it.
            self.__dict__["expensive_computation"] = result
            return result


print("Concurrent access to cached properties")
print("=" * 38)

n_threads = 5

# 1) No lock: all threads read the cold property at the same moment.
shared = SharedComputation("ConcurrentTest")
barrier = threading.Barrier(n_threads)

def read_after_barrier(obj):
    barrier.wait()
    return obj.expensive_computation

with ThreadPoolExecutor(max_workers=n_threads) as executor:
    thread_values = list(executor.map(read_after_barrier, [shared] * n_threads))

stored = shared.__dict__["expensive_computation"]
print(f"Getter ran {len(shared.computed_values)} time(s) for {n_threads} concurrent readers "
      "(several threads may each compute the value)")
print(f"Distinct values seen by the threads: {len(set(thread_values))}")
assert 1 <= len(shared.computed_values) <= n_threads, shared.computed_values
assert set(thread_values) <= set(shared.computed_values), (thread_values, shared.computed_values)
assert stored in shared.computed_values, stored
# Once the threads are done, the instance holds one value and every read returns it.
assert all(shared.expensive_computation is stored for _ in range(3))

# Mixed reads after the value is cached: every thread sees the same values.
def read_both(obj):
    return obj.expensive_computation, obj.derived_property

with ThreadPoolExecutor(max_workers=4) as executor:
    mixed = list(executor.map(read_both, [shared] * 4))
expected_derived = stored * 2 + len("ConcurrentTest")
assert mixed == [(stored, expected_derived)] * 4, mixed
print(f"Mixed reads after caching: expensive={stored}, derived={expected_derived} in all 4 threads")

# 2) With a lock in the getter: the computation runs once and all threads get one value.
locked = LockedComputation()
barrier = threading.Barrier(n_threads)
with ThreadPoolExecutor(max_workers=n_threads) as executor:
    locked_values = list(executor.map(read_after_barrier, [locked] * n_threads))

print(f"With a lock: computed {locked.compute_count} time(s), values seen: {set(locked_values)}")
assert locked.compute_count == 1, locked.compute_count
assert len(set(locked_values)) == 1, locked_values
assert locked.expensive_computation is locked.__dict__["expensive_computation"]


In [ ]:
# Example 4: Performance and where the cached value is stored
import propcache
import time
import sys
import gc
import weakref
from functools import lru_cache

class PerformanceComparison:
    """Compare cached_property performance with alternatives"""
    
    def __init__(self, data_size=1000):
        self.data_size = data_size
        self.raw_data = list(range(data_size))
        self._regular_property_calls = 0
        self._cached_property_calls = 0
        self._lru_cache_calls = 0
    
    # Regular property (no caching)
    @property 
    def uncached_expensive_operation(self):
        """Expensive operation without caching"""
        self._regular_property_calls += 1
        # Simulate expensive computation
        result = sum(x * x for x in self.raw_data)
        time.sleep(0.01)  # Simulate I/O or complex calculation
        return result
    
    # propcache cached_property
    @propcache.cached_property
    def cached_expensive_operation(self):
        """Same operation with propcache caching"""
        self._cached_property_calls += 1
        result = sum(x * x for x in self.raw_data)
        time.sleep(0.01)  # Simulate I/O or complex calculation
        return result
    
    # lru_cache comparison (method caching)
    @lru_cache(maxsize=1)
    def lru_cached_operation(self):
        """Same operation with lru_cache"""
        self._lru_cache_calls += 1
        result = sum(x * x for x in self.raw_data)
        time.sleep(0.01)  # Simulate I/O or complex calculation  
        return result
    
    @propcache.cached_property
    def memory_intensive_property(self):
        """Property that creates large data structures"""
        # Create a large dictionary for memory testing
        return {i: f"value_{i}_{'x' * 100}" for i in range(min(100, self.data_size))}

def time_operation(func, iterations=5):
    """Time a function over multiple iterations"""
    times = []
    for _ in range(iterations):
        start = time.time()
        func()
        times.append(time.time() - start)
    return times

print("Performance and cache storage")
print("=" * 28)

# Create performance test object
perf_obj = PerformanceComparison(data_size=500)

# Test uncached performance
print("Testing uncached property (5 accesses):")
uncached_times = time_operation(lambda: perf_obj.uncached_expensive_operation, 5)
print(f"  Times: {[f'{t:.4f}s' for t in uncached_times]}")
print(f"  Average: {sum(uncached_times)/len(uncached_times):.4f}s")
print(f"  Total calls: {perf_obj._regular_property_calls}")
assert perf_obj._regular_property_calls == 5, perf_obj._regular_property_calls

# Test cached_property performance
print(f"\nTesting cached_property (5 accesses):")
cached_times = time_operation(lambda: perf_obj.cached_expensive_operation, 5)
print(f"  Times: {[f'{t:.4f}s' for t in cached_times]}")
print(f"  Average: {sum(cached_times)/len(cached_times):.4f}s")
print(f"  Speedup: {sum(uncached_times)/sum(cached_times):.1f}x")
print(f"  Total calls: {perf_obj._cached_property_calls}")
assert perf_obj._cached_property_calls == 1, perf_obj._cached_property_calls

# Test lru_cache performance
print(f"\nTesting lru_cache (5 accesses):")
lru_times = time_operation(lambda: perf_obj.lru_cached_operation(), 5)
print(f"  Times: {[f'{t:.4f}s' for t in lru_times]}")
print(f"  Average: {sum(lru_times)/len(lru_times):.4f}s")
print(f"  Total calls: {perf_obj._lru_cache_calls}")
assert perf_obj._lru_cache_calls == 1, perf_obj._lru_cache_calls

# Cache storage: the value lands in the instance __dict__ on first access, not before
print(f"\nCache storage:")
name = "memory_intensive_property"
attrs_before = set(vars(perf_obj))
assert name not in attrs_before, sorted(attrs_before)

memory_data = perf_obj.memory_intensive_property
attrs_after = set(vars(perf_obj))
print(f"  New instance attributes after first access: {sorted(attrs_after - attrs_before)}")
print(f"  Cached dict holds {len(memory_data)} entries ({sys.getsizeof(memory_data):,} bytes for the dict itself)")
assert attrs_after - attrs_before == {name}, sorted(attrs_after - attrs_before)
assert vars(perf_obj)[name] is memory_data
assert perf_obj.memory_intensive_property is memory_data  # second access returns the same object
assert len(memory_data) == 100

# Test multiple instances to show per-instance caching
print(f"\nTesting per-instance caching:")
instances = [PerformanceComparison(data_size=100) for _ in range(3)]

for i, instance in enumerate(instances):
    # First access should compute
    start_time = time.time()
    result = instance.cached_expensive_operation
    first_access_time = time.time() - start_time
    
    # Second access should be cached
    start_time = time.time()
    result2 = instance.cached_expensive_operation
    second_access_time = time.time() - start_time
    
    print(f"  Instance {i+1}: First={first_access_time:.4f}s, Second={second_access_time:.4f}s")
    print(f"    Same result? {result == result2}")
    assert result == result2 == sum(x * x for x in range(100)), (result, result2)
    assert instance._cached_property_calls == 1, instance._cached_property_calls

# The cache lives in each instance, so it does not keep the instance alive
print(f"\nGarbage collection and cleanup:")
first_ref = weakref.ref(instances[0])
del instances, instance
gc.collect()
assert first_ref() is None, "instance with a cached value was not freed"
print(f"  Instances deleted and garbage collected")

In [ ]:
# Example 5: Advanced patterns and production use cases
import propcache
import time
import json
import hashlib
from datetime import datetime, timedelta
from typing import Dict, Any, Optional

class ProductionDataProcessor:
    """Production-ready class with advanced cached property patterns"""
    
    def __init__(self, config: Dict[str, Any]):
        self.config = config
        self._cache_stats = {
            'hits': 0,
            'misses': 0,
            'computations': 0
        }
    
    @propcache.cached_property
    def configuration_hash(self) -> str:
        """Generate hash of configuration for cache invalidation"""
        config_str = json.dumps(self.config, sort_keys=True)
        return hashlib.md5(config_str.encode()).hexdigest()[:8]
    
    @propcache.cached_property
    def database_connection_pool(self) -> Dict[str, Any]:
        """Simulate expensive database connection pool setup"""
        self._cache_stats['computations'] += 1
        print(f"Initializing database connection pool...")
        
        # Simulate connection setup time
        time.sleep(0.1)
        
        pool_config = {
            'host': self.config.get('db_host', 'localhost'),
            'port': self.config.get('db_port', 5432),
            'pool_size': self.config.get('pool_size', 10),
            'initialized_at': datetime.now().isoformat(),
            'config_hash': self.configuration_hash
        }
        
        return pool_config
    
    @propcache.cached_property
    def api_client_config(self) -> Dict[str, Any]:
        """API client configuration with timeout and retry settings"""
        self._cache_stats['computations'] += 1
        print(f"Building API client configuration...")
        
        return {
            'base_url': self.config.get('api_base_url', 'https://api.example.com'),
            'timeout': self.config.get('api_timeout', 30),
            'max_retries': self.config.get('max_retries', 3),
            'auth_token': self._generate_auth_token(),
            'config_hash': self.configuration_hash
        }
    
    def _generate_auth_token(self) -> str:
        """Generate authentication token (expensive operation)"""
        time.sleep(0.05)  # Simulate token generation
        return f"token_{self.configuration_hash}_{int(time.time())}"
    
    @propcache.cached_property
    def processed_data_schema(self) -> Dict[str, Any]:
        """Complex data schema processing"""
        self._cache_stats['computations'] += 1
        print(f"Processing data schema...")
        
        # Simulate complex schema processing
        schema_rules = self.config.get('schema_rules', [])
        processed_schema = {
            'fields': {},
            'validators': [],
            'transformers': []
        }
        
        for rule in schema_rules:
            field_name = rule.get('field')
            field_type = rule.get('type', 'string')
            
            processed_schema['fields'][field_name] = {
                'type': field_type,
                'required': rule.get('required', False),
                'default': rule.get('default')
            }
            
            if rule.get('validator'):
                processed_schema['validators'].append({
                    'field': field_name,
                    'rule': rule['validator']
                })
        
        return processed_schema
    
    def update_config(self, new_config: Dict[str, Any]):
        """Update configuration and invalidate relevant caches"""
        old_hash = self.__dict__.get('configuration_hash')  # cached value, if any
        
        # Update configuration
        self.config.update(new_config)
        
        # Invalidate configuration hash to force recomputation
        self.__dict__.pop('configuration_hash', None)
        
        # Get new hash
        new_hash = self.configuration_hash
        
        # Invalidate dependent caches if config actually changed
        if old_hash != new_hash:
            dependent_caches = [
                'database_connection_pool',
                'api_client_config',
                'processed_data_schema'
            ]
            
            for name in dependent_caches:
                if self.__dict__.pop(name, None) is not None:
                    print(f"Invalidated cache: {name}")
    
    def get_cache_stats(self) -> Dict[str, Any]:
        """Get caching statistics"""
        cached_properties = sorted(
            name for name in vars(self)
            if isinstance(type(self).__dict__.get(name), propcache.cached_property)
        )
        
        return {
            **self._cache_stats,
            'cached_properties': cached_properties,
            'total_cached': len(cached_properties)
        }
    
    def warmup_caches(self):
        """Pre-compute all cached properties for better performance"""
        print("Warming up caches...")
        properties_to_warmup = [
            'configuration_hash',
            'database_connection_pool', 
            'api_client_config',
            'processed_data_schema'
        ]
        
        for prop_name in properties_to_warmup:
            start_time = time.time()
            getattr(self, prop_name)
            duration = time.time() - start_time
            print(f"  {prop_name}: {duration:.4f}s")

print("Advanced patterns and production use cases")
print("=" * 42)

# Create production processor with complex config
production_config = {
    'db_host': 'prod-db.example.com',
    'db_port': 5432,
    'pool_size': 20,
    'api_base_url': 'https://prod-api.example.com',
    'api_timeout': 60,
    'max_retries': 5,
    'schema_rules': [
        {'field': 'user_id', 'type': 'integer', 'required': True},
        {'field': 'email', 'type': 'string', 'required': True, 'validator': 'email'},
        {'field': 'age', 'type': 'integer', 'default': 0},
        {'field': 'preferences', 'type': 'object', 'default': {}}
    ]
}

processor = ProductionDataProcessor(production_config)

# Test cache warming
print("Initial cache warmup:")
processor.warmup_caches()

# Show initial cache stats
print(f"\nCache stats after warmup:")
stats = processor.get_cache_stats()
print(f"  Computations: {stats['computations']}")
print(f"  Cached properties: {stats['cached_properties']}")
all_cached = ['api_client_config', 'configuration_hash', 'database_connection_pool', 'processed_data_schema']
assert stats['computations'] == 3, stats  # configuration_hash does not count itself
assert stats['cached_properties'] == all_cached, stats

# Test repeated access (should hit cache)
print(f"\nTesting cache hits:")
start_time = time.time()
db_pool1 = processor.database_connection_pool
api_config1 = processor.api_client_config
schema1 = processor.processed_data_schema
access_time = time.time() - start_time
print(f"  Second access time: {access_time:.4f}s (should be fast)")
assert processor.get_cache_stats()['computations'] == 3  # all three were cache hits

# Test configuration update and invalidation
print(f"\nTesting configuration update and cache invalidation:")
processor.update_config({
    'pool_size': 30,
    'api_timeout': 90,
    'schema_rules': production_config['schema_rules'] + [
        {'field': 'created_at', 'type': 'datetime', 'required': True}
    ]
})

assert processor.get_cache_stats()['cached_properties'] == ['configuration_hash']  # dependents dropped

# Access properties after update (should recompute some)
print(f"Accessing properties after config update:")
db_pool2 = processor.database_connection_pool
api_config2 = processor.api_client_config
schema2 = processor.processed_data_schema

# Compare results
print(f"\nComparing results after update:")
print(f"  DB pool changed? {db_pool1 != db_pool2}")
print(f"  API config changed? {api_config1 != api_config2}")
print(f"  Schema changed? {schema1 != schema2}")
assert db_pool2['pool_size'] == 30 and db_pool2['config_hash'] != db_pool1['config_hash']
assert api_config2['timeout'] == 90 and 'created_at' in schema2['fields']

# Final cache stats
final_stats = processor.get_cache_stats()
print(f"\nFinal cache statistics:")
print(f"  Total computations: {final_stats['computations']}")
print(f"  Currently cached: {final_stats['total_cached']} properties")
print(f"  Cached property names: {final_stats['cached_properties']}")
assert final_stats['computations'] == 6 and final_stats['cached_properties'] == all_cached, final_stats

print("✓ Advanced patterns and production use cases verified!")